# --- SECTION: INTRODUCTION ---
🥉 ElectroFlow: Building the Bronze Layer (Learning Edition)
In this notebook, we'll build the ingestion logic from scratch.
Follow the instructions in the comments and fill in the code!

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import *
import uuid
import os

# We use os.getcwd() to find our position in the Workspace.
# We then strip the current folder '01_bronze' to find the Repo Root.
repo_root = os.getcwd().split("/01_bronze")[0]

# Base landing zone path in Unity Catalog Volume
landing_base = "/Volumes/dev/electroflow_pipeline/landing_data"

# Configuration for source files
source_files = [
    {"name": "customers", "file": "customers.csv", "format": "csv"},
    {"name": "products",  "file": "products.csv",  "format": "csv"},
    {"name": "orders",    "file": "orders.json",    "format": "json"},
    {"name": "payments",  "file": "order_payments.csv", "format": "csv"},
    {"name": "coupons",   "file": "coupons.csv",   "format": "csv"}
]

In [0]:
#the ingestions function
def load_raw_to_bronze(source_path, target_table, file_format='csv'):
    print(f"Loading {source_path} to {target_table}")

    #read the raw data
    if file_format == "csv":
        df = spark.read.format("csv")\
            .option("header","true")\
            .option("inferschema","true")\
            .load(source_path)
    elif file_format == "json":
        df = spark.read.format('json')\
            .option("inferschema","true")\
            .option("multiline", "true")\
            .load(source_path) 

    #enrich the data
    df_enriched = df.select("*", F.col("_metadata.file_path").alias("_source_file_path")) \
                    .withColumn("_ingestion_timestamp", F.current_timestamp()) \
                    .withColumn("_ingestion_job_id", F.lit(str(uuid.uuid4())))

    #write to bronze table
    # Fix: Use a Unity Catalog volume location for Delta tables
    target_path = "/Volumes/dev/electroflow_pipeline/bronze_data/" + target_table
    df_enriched.write.format("delta")\
        .mode("overwrite")\
        .save(target_path)

    print (f"Finished ingestion to {target_table}")
    return df_enriched

# --- Execution
for config in source_files:
    source_path = f"{landing_base}/{config['file']}"
    target_table = f"bronze_{config['name']}"
    load_raw_to_bronze(source_path, target_table, file_format=config['format'])